# Notebook 01 — Modelos LLM e NLP com Hugging Face

**Objetivo:** Demonstrar dominio do ecossistema Hugging Face com tarefas NLP aplicadas ao dominio de bulas medicas.

**Rubrica 1:** Construir aplicacoes NLP com LLMs e ecossistema Hugging Face (5 itens).

## 2.1 Setup e Imports

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

import torch
from transformers import pipeline, AutoModel, AutoTokenizer, AutoModelForQuestionAnswering
from transformers import AutoModelForSeq2SeqLM
from scripts.config import DEVICE, NER_MODEL, EMBEDDING_MODEL

## 2.2 Carregando Modelo com AutoModel + AutoTokenizer

Demonstracao: `AutoTokenizer`, `AutoModel`, inspecao de hidden states.

**Modelo:** `pucpr/clinicalnerpt-chemical` — BERT para NER clinico em portugues.

In [ ]:
from scripts.ner import MedicationNER

# Carrega NER com aggregation_strategy="simple" (agrega B-/I- em nomes completos)
ner = MedicationNER()

consultas = [
    "Posso tomar warfarina com aspirina?",
    "Amoxicilina e Metotrexato juntos e seguro?",
    "Dipirona com Paracetamol, pode?",
    "AAS Protect e Ibuprofeno causam sangramento?",
]

print("NER — Extracao de Medicamentos")
print("=" * 55)
for q in consultas:
    meds = ner.extrair_medicamentos(q)
    pares = ner.extrair_pares(q)
    print(f"\nQuery: {q}")
    print(f"  Extraidos: {meds}")
    if pares:
        print(f"  Pares: {pares}")



**Observacoes:**
- Tokenizador BERT usa **WordPiece**: palavras frequentes → tokens unicos, raras → sub-tokens
- Limite: **512 tokens**
- `last_hidden_state`: embedding contextualizado de cada token

## 2.3 Pipeline: sentiment-analysis em Frases Clinicas

Modelo generico em dominio especializado — demonstrando limitacoes que motivam fine-tuning.

In [ ]:
from scripts.classifier import InteractionClassifier

clf = InteractionClassifier()

casos = [
    ("warfarina", "aspirina",
     "O uso concomitante e contraindicado devido ao risco de arritmia fatal."),
    ("amoxicilina", "metotrexato",
     "A administracao concomitante pode aumentar a toxicidade do Metotrexato."),
    ("paracetamol", "alcool",
     "Recomenda-se evitar o uso de alcool durante o tratamento."),
]

print("Classificador de Interacoes — BioBERTpt fine-tuned")
print("=" * 55)
for alvo, outro, contexto in casos:
    resultado = clf.classificar(alvo, outro, contexto)
    nome = ["SEM interacao", "INTERACAO LEVE", "INTERACAO GRAVE"][resultado["classe"]]
    print(f"\n[{nome}] confianca={resultado['confianca']:.1%}")
    print(f"  {alvo} + {outro}")
    print(f"  Contexto: {contexto[:60]}...")



**Analise:** O modelo classifica por tom emocional, nao por significado clinico. Precisamos de modelos treinados em dominio clinico.

## 2.4 Pipeline: NER com clinicalnerpt-chemical

**NER (token classification):** identifica nomes de medicamentos — principios ativos e nomes comerciais.

In [ ]:
ner = pipeline("ner", model=NER_MODEL, aggregation_strategy="simple",
             device=0 if DEVICE == "cuda" else -1)

trecho = (
    "A probenecida reduz a secrecao tubular renal da amoxicilina. "
    "No uso concomitante com amoxicilina, pode haver aumento dos niveis "
    "de amoxicilina no sangue. A administracao concomitante de alopurinol "
    "durante o tratamento com amoxicilina pode aumentar a probabilidade "
    "de reacoes alergicas da pele. Existem casos raros de INR aumentada "
    "em pacientes mantidos com acenocumarol ou varfarina."
)

entidades = ner(trecho)
print('NER — Processando trecho clinico...')
for ent in entidades:
    print(f'{ent["word"]:<25} {ent["score"]:>8.3f} [{ent["start"]}:{ent["end"]}]')

unicos = list(set(ent["word"] for ent in entidades))
print(f'\nMedicamentos identificados ({len(unicos)}): {unicos}')
print(f"\nMedicamentos identificados ({len(unicos)}): {unicos}")

**Analise:** Modelo identifica corretamente amoxicilina, probenecida, alopurinol, acenocumarol, varfarina. Encoder-only (BERT) ideal para NER. Este sera o primeiro estagio do pipeline RAG.

## 2.5 Pipeline: text-generation com GPT-2 Portugues

**Decoder-only:** geracao autoregressiva token por token. Demonstracao de alucinacao.

In [ ]:
gerador = pipeline("text-generation", model="pierreguillou/gpt2-small-portuguese")

prompt = "Interacao entre Amoxicilina e Ibuprofeno:"

r1 = gerador(prompt, max_length=80, do_sample=True, temperature=0.9)
print("=== temperature=0.9 (criativa) ===")
print(r1[0]["generated_text"])

r2 = gerador(prompt, max_length=80, do_sample=True, temperature=0.3, top_k=20)
print("\n=== temperature=0.3 + top_k=20 (conservadora) ===")
print(r2[0]["generated_text"])

**Analise:** GPT-2 gera texto fluente mas **alucina** informacoes — nao tem conhecimento medico real. Decoder-only (geracao) vs Encoder-only (compreensao).

## 2.6 Pipeline: fill-mask com BERT Portugues

Revela conhecimento latente do modelo ao prever tokens mascarados.

In [ ]:
unmasker = pipeline("fill-mask", model=EMBEDDING_MODEL)

print("=== Teste 1: contexto clinico ===")
r1 = unmasker("O uso concomitante de Amoxicilina com Metotrexato e [MASK] devido ao risco de toxicidade.")
for r in r1:
    print(f"  {r["token_str"]:>12} | score={r["score"]:.4f}")

print("\n=== Teste 2: contexto de seguranca ===")
r2 = unmasker("Nao ha interacoes conhecidas. O medicamento e [MASK] para uso.")
for r in r2[:3]:
    print(f"  {r["token_str"]:>12} | score={r["score"]:.4f}")

**Analise:** BERT preve 'contraindicado' no contexto de toxicidade, 'seguro' no contexto positivo. Atencao **bidirecional** permite preencher lacunas. Este modelo sera usado para gerar embeddings na Fase 6.

## 2.7 Sumarizacao com BART (via AutoModel)

**Encoder-decoder:** sumarizacao abstrativa. A partir do Transformers 5.x, usamos `AutoModelForSeq2SeqLM` diretamente.

In [ ]:
print('Carregando tokenizer e modelo de sumarizacao (facebook/bart-large-cnn)...')
summ_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
summ_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(DEVICE)

texto = (
    "Miopatia pode ocorrer em pacientes que usam Zarator, sendo mais frequentes "
    "naqueles que usam tambem ciclosporina, fibratos, niacina ou antifungicos "
    "azolicos. A administracao concomitante com medicamentos inibidores do "
    "citocromo P450 3A4 (ciclosporina, eritromicina/claritromicina, inibidores "
    "da protease) pode alterar a quantidade de atorvastatina no sangue. Sao "
    "conhecidas interacoes com antiacidos, colestipol, contraceptivos orais, "
    "varfarina, acido fusidico."
)

inputs = summ_tokenizer(texto, max_length=1024, truncation=True, return_tensors="pt").to(DEVICE)
summary_ids = summ_model.generate(inputs["input_ids"], max_length=80, min_length=30,
                                   num_beams=4, early_stopping=True)
resumo = summ_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"Original ({len(texto.split())} palavras):")
print(texto[:200] + "...")
print(f"\nResumo ({len(resumo.split())} palavras):")
print(resumo)

**Analise:** BART e ~4x maior que BERT (406M vs 110M). Treinado em ingles → qualidade limitada em portugues. `model.generate()` da controle sobre beams, early stopping.

## 2.8 Question Answering com BERT em Portugues (via AutoModel)

**QA extrativo:** encontra span de resposta no contexto. A partir do Transformers 5.x, usamos `AutoModelForQuestionAnswering` diretamente.

In [ ]:
qa_model_id = "pierreguillou/bert-base-cased-squad-v1.1-portuguese"
print('Carregando modelo de QA em Portugues...')
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_id)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_id).to(DEVICE)

contexto = (
    "A probenecida reduz a secrecao tubular renal da amoxicilina. "
    "A administracao concomitante de alopurinol durante o tratamento "
    "com amoxicilina pode aumentar a probabilidade de reacoes alergicas "
    "da pele. Existem casos raros de INR aumentada em pacientes mantidos "
    "com acenocumarol ou varfarina, ao receberem um curso de tratamento "
    "com amoxicilina. Se a coadministracao e necessaria, o tempo de "
    "protrombina ou INR deve ser cuidadosamente monitorado."
)

perguntas = [
    "Quais medicamentos interagem com Amoxicilina?",
    "Qual o risco de tomar Amoxicilina com Varfarina?",
    "Qual a dose maxima recomendada de Amoxicilina?",
]

for i, pergunta in enumerate(perguntas, 1):
    inputs = qa_tokenizer(pergunta, contexto, max_length=512,
                          truncation=True, return_tensors="pt").to(DEVICE)
    outputs = qa_model(**inputs)
    start_idx = outputs.start_logits.argmax()
    end_idx = outputs.end_logits.argmax()
    answer_ids = inputs["input_ids"][0][start_idx:end_idx+1]
    answer = qa_tokenizer.decode(answer_ids)
    score = (outputs.start_logits.max() + outputs.end_logits.max()).item()
    print(f"P{i}: {answer} (score: {score:.1f})")

**Analise:** P1 extrai medicamentos corretamente. P2 encontra 'INR aumentada' (associacao indireta). P3 retorna algo com score baixo — QA extrativo **sempre retorna um span**, mesmo sem resposta. **Licao para o RAG:** usaremos geracao fundamentada (LLM + contexto), que pode dizer 'sem informacao'.

## 2.9 Tabela Comparativa de Modelos e Arquiteturas

| Modelo | Arquitetura | Param. | Tarefa | Limite | Dominio |
|---|---|---|---|---|---|
| `clinicalnerpt-chemical` | BERT (encoder-only) | 110M | NER | 512 | Clinico PT |
| `biobertpt-all` | BERT (encoder-only) | 110M | Classificacao | 512 | Biomedico PT |
| `bart-large-cnn` | BART (encoder-decoder) | 406M | Sumarizacao | 1024 | Generico EN |
| `gpt2-small-portuguese` | GPT-2 (decoder-only) | 124M | Geracao | 1024 | Generico PT |
| `bert-base-portuguese-cased` | BERT (encoder-only) | 110M | Fill-mask / Embeddings | 512 | Generico PT |

### Diferenças entre Arquiteturas

- **Encoder-only (BERT):** Atencao bidirecional — cada token ve contexto completo. Ideal para compreensao: NER, classificacao, QA extrativo, embeddings.
- **Decoder-only (GPT-2):** Atencao unidirecional/causal — cada token so ve contexto anterior. Ideal para geracao: chatbots, completamento de texto.
- **Encoder-decoder (BART):** Combina ambos — encoder processa entrada, decoder gera saida. Ideal para traducao e sumarizacao.

### Pipeline vs Inferencia Manual

- `pipeline()`: rapido, encapsula tokenizacao + modelo + post-processamento. Ideal para prototipagem.
- Inferencia manual (`AutoModel` + `tokenizer`): controle total sobre tokenizacao, GPU, batched inference. Necessario para fine-tuning e producao.
- A partir do Transformers 5.x, alguns pipelines foram descontinuados (ex: `summarization`, `question-answering`) — usar `AutoModel` diretamente.

## 2.10 Conclusao: O que Aprendemos

### Tarefas uteis para o detector de interacoes:

| Tarefa | Aplicacao no Projeto | Fase |
|---|---|---|
| **NER** | Extrair medicamentos da consulta do usuario | Fase 8 (RAG) |
| **Classificacao** | Classificar interacao (0/1/2) com BioBERTpt fine-tuned | Fase 4 |
| **Embeddings** | Indexar chunks no ChromaDB para busca vetorial | Fase 6 |
| **Geracao (LLM)** | Produzir resposta final fundamentada nos chunks | Fase 8 (RAG) |

### Proximos passos:
- **Fase 3:** Anotar dataset de treino (~1.500 pares) com weak supervision
- **Fase 4:** Fine-tuning do BioBERTpt para classificacao de interacoes
- **Fase 5:** Prompt engineering com LLMs (zero-shot, few-shot, CoT)